In [ ]:
import sys
from pathlib import Path

# Notebooks run from their own folder; the packages live in the project root.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT))

from config import load_settings
from dataset.DataLoader import create_dataloaders
from finetuning import ModelCatalog
from finetuning.FineTuner import fine_tune
from finetuning.FineTuneStrategy import FineTuneStrategy
from utils import model_utils

settings = load_settings()

# True: check that the notebook runs end to end in minutes (16 images per class, 1 epoch). The numbers mean nothing.
QUICK_CHECK = False
SAMPLES_PER_CLASS = 16 if QUICK_CHECK else None


def epochs(n: int) -> int:
    return 1 if QUICK_CHECK else n

## Default model catalog
For the default catalog I have provided the agent with 5 models. 2 of these are vision transformers and the rest are classical deep convolutional nets.

In [ ]:
catalog = ModelCatalog.get_default_catalog()
print(catalog)

As you can see in the above cell, the param count ranges from 5.5M to 86.6M. The goal was to provide the model with different tools, each capable of achieving differing levels of accuracy with different parameter counts.

In [ ]:
models_data = [
    catalog.get_model_data("vit_b_16"),
    catalog.get_model_data("swin_s"),
    catalog.get_model_data("resnet50"),
    catalog.get_model_data("densenet161"),
    catalog.get_model_data("mobilenet_v3_large")
]
models = [model_utils.get_model(model) for model in models_data]
print(models[0])

In [ ]:
train_loader, val_loader, test_loader = create_dataloaders(
    settings.dataset_path, batch_size=8, samples_per_class=SAMPLES_PER_CLASS
)

The outputs these cells used to carry were removed. They came from code with the LoRA bugs C1 and C2 (see `goals.md`), so the ViT LoRA run classified through a frozen random head, and none of them used a fair evaluation protocol. Comparable numbers come from G3.

In [ ]:
vit_lora = fine_tune(models[0],
                FineTuneStrategy.LORA,
                train_loader=train_loader,
                val_loader=val_loader,
                model_dir=settings.model_dir,
                num_epochs=epochs(15),
                )
vit_lora

In [ ]:
resnet_full_ft = fine_tune(models[2],
                          strategy=FineTuneStrategy.FULL_FT,
                          train_loader=train_loader,
                          val_loader=val_loader,
                          model_dir=settings.model_dir,
                          num_epochs=epochs(10))
resnet_full_ft

In [ ]:
mobilenet_probe = fine_tune(models[4],
                            strategy=FineTuneStrategy.LINEAR_PROBE,
                            train_loader=train_loader,
                            val_loader=val_loader,
                            model_dir=settings.model_dir,
                            num_epochs=epochs(10))
mobilenet_probe

## Software Architecture
To implement the `fine_tune` method, I have used the adapter pattern to create `BreastCancerModel`. The main part of this package is the `BreastCancerModel`. So when the agent calls `get_model` from the `utils` package, it should provide the `ModelData` of the respective model. This `ModelData` contains an adapter. The contains vital information about each model including :
- **classifying head**
- **replacement method for default classifying head**
- **lora `target_modules`**
The constructor of `BreastCancerModel` uses this adapter to set up each model with appropriate architecture.

There are 3 finetuning strategies defined. `FULL_FT`, `LINEAR_PROBE` and `LORA`. Based on the selected strategy parts of the provided model are frozen. The agent can select weight decay, learning rate and number of epochs for finetuning. After finetuning is completed, The best epoch is saved (validation accuracy is the decision-making factor) in a unique path and the result is saved in the memory module (for the agent) as a `FineTuneResult` object. This object stores important information about the result of fine-tuning operation, including the per class confusion matrix, accuracy, precision, recall and f1 scores.